# M02 — LightGBM: Gradient Boosting at Scale

## Why LightGBM is different from XGBoost

LightGBM makes two algorithmic innovations over XGBoost:

**1. Leaf-wise tree growth (vs level-wise in XGBoost)**
- XGBoost grows trees level by level — all nodes at depth $d$ before going to $d+1$
- LightGBM grows the leaf with the highest gain first — regardless of depth
- Result: deeper, less symmetric trees that fit the data more aggressively
- Risk: more likely to overfit → control with `num_leaves` (not `max_depth`)

**2. Histogram-based splitting**
- Instead of sorting all values to find the best split threshold, LightGBM bins continuous features into $k$ buckets (default 255)
- Finding the best split is now $O(k)$ instead of $O(n)$
- This is why LightGBM is much faster on large datasets

**3. GOSS and EFB (bonus optimizations)**
- **GOSS** (Gradient-based One-Side Sampling): only keep samples with large gradients (high error) + random sample of small-gradient samples. Reduces data without losing important signal.
- **EFB** (Exclusive Feature Bundling): bundles mutually exclusive sparse features together. Reduces feature dimensionality.

## Key parameters (different from XGBoost)

| Parameter | LightGBM | XGBoost equivalent |
|---|---|---|
| Tree complexity | `num_leaves` | `max_depth` |
| Regularization | `min_child_samples` | `min_child_weight` |
| Feature sampling | `feature_fraction` | `colsample_bytree` |
| Row sampling | `bagging_fraction` | `subsample` |
| L1 | `reg_alpha` | `reg_alpha` |
| L2 | `reg_lambda` | `reg_lambda` |

**Reference:** [LightGBM docs](https://lightgbm.readthedocs.io/en/stable/)


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Datasets
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')
credit['target'] = (credit['class'] == 'good').astype(int)

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]

cat_cols = credit.select_dtypes(include='object').columns.drop('class').tolist()
num_cols = credit.select_dtypes(include='number').columns.drop('target').tolist()
X = credit[num_cols + cat_cols]
y = credit['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
feature_names = num_cols + cat_cols

print(f"LightGBM version: {lgb.__version__}")
print(f"Credit: {credit.shape} | Features: {len(feature_names)}")

---
## Exercise 1 — Native LightGBM API: Dataset and Train

**Task:** Use LightGBM's native API (not sklearn wrapper) — this is what practitioners use for maximum control.

1. Create `lgb.Dataset` for train and validation sets with feature names.
2. Define params dict: `objective='binary'`, `metric='auc'`, `num_leaves=31`, `learning_rate=0.05`, `n_estimators=200`, `verbosity=-1`.
3. Train with `lgb.train()`, passing a watchlist and enabling early stopping (patience=15).
4. Extract: `best_iteration`, `best_score`.
5. Predict and compute test AUC.

**Key difference from XGBoost:** LightGBM's `predict()` returns probabilities directly (not log-odds). Note this.

In [ ]:
# YOUR CODE HERE
lgb_model = None
best_iteration = None
best_score = None
test_auc = None

In [ ]:
# --- ASSERTIONS ---
assert lgb_model is not None
assert best_iteration is not None and best_iteration > 0
assert test_auc > 0.65, f"AUC must be > 0.65, got {test_auc:.4f}"
# Predictions must be probabilities [0, 1]
preds = lgb_model.predict(X_test_enc)
assert preds.min() >= 0 and preds.max() <= 1, "LightGBM native predict returns probabilities"
print(f"✓ Exercise 1 passed — Best iter: {best_iteration} | Test AUC: {test_auc:.4f}")

---
## Exercise 2 — num_leaves vs max_depth: The Core Trade-off

**Concept:** `num_leaves` is the primary complexity parameter in LightGBM (not `max_depth`).
- A balanced binary tree of depth $d$ has $2^d$ leaves
- `num_leaves=31` ≈ depth 5 for a balanced tree
- But LightGBM trees are NOT balanced — a tree with `num_leaves=31` might be depth 20
- Rule of thumb: `num_leaves < 2^max_depth` to avoid overfitting

**Task:**
1. Train models with `num_leaves` in `[8, 16, 31, 64, 128, 256]`, keeping `max_depth=-1` (no limit).
2. For each: record `train_auc`, `test_auc`, `overfit_gap = train_auc - test_auc`.
3. Build `leaves_results`: DataFrame with these metrics.
4. Identify the `optimal_num_leaves`: the value with best test_auc.
5. In markdown: explain what you observe about overfitting as num_leaves increases.

In [ ]:
# YOUR CODE HERE
leaves_results = None
optimal_num_leaves = None

In [ ]:
# --- ASSERTIONS ---
assert leaves_results is not None and len(leaves_results) == 6
assert 'train_auc' in leaves_results.columns
assert 'test_auc' in leaves_results.columns
assert 'overfit_gap' in leaves_results.columns
assert optimal_num_leaves in [8, 16, 31, 64, 128, 256]
# Overfit gap should generally increase with num_leaves
gaps = leaves_results['overfit_gap'].values
assert gaps[-1] > gaps[0], "Overfit gap must increase as num_leaves grows"
print(f"✓ Exercise 2 passed — Optimal num_leaves: {optimal_num_leaves}")
print(leaves_results.to_string(index=False))

**Observation:** *(What do you see about the train/test gap as num_leaves increases? What is the optimal range for this dataset?)*

---
## Exercise 3 — Categorical Feature Support

**LightGBM's killer feature:** it handles categorical features natively without one-hot encoding — using an optimal split algorithm for categories.

**How it works:** For a categorical feature with $k$ categories, LightGBM finds the optimal binary partition of categories into two groups by sorting them by gradient statistics. This is $O(k \log k)$ vs $O(2^k)$ for brute force.

1. Pass raw categorical columns to LightGBM using `categorical_feature` parameter (no OrdinalEncoder needed — but dtypes must be int or `'category'`).
2. Convert categoricals to `pd.Categorical` dtype.
3. Train with `categorical_feature='auto'` and compare to the OrdinalEncoded version from Exercise 1.
4. Compare: test AUC, training time, number of splits per categorical feature.
5. Return `categorical_comparison`: DataFrame comparing both approaches.

In [ ]:
import time

# Prepare data with pandas Categorical dtype
X_cat = X.copy()
for col in cat_cols:
    X_cat[col] = pd.Categorical(X_cat[col])

X_train_cat, X_test_cat = train_test_split(X_cat, test_size=0.2, random_state=42)
y_train_cat = y_train if len(y_train) == len(X_train_cat) else y.iloc[X_train_cat.index]

# YOUR CODE HERE
categorical_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert categorical_comparison is not None
assert len(categorical_comparison) == 2
assert 'test_auc' in categorical_comparison.columns
assert (categorical_comparison['test_auc'] > 0.60).all()
print("✓ Exercise 3 passed")
print(categorical_comparison.to_string(index=False))

---
## Exercise 4 — DART Boosting Mode

**Concept:** DART (Dropouts meet Multiple Additive Regression Trees) applies the dropout idea from neural networks to gradient boosting:
- During each boosting round, randomly drop a fraction of existing trees
- The new tree compensates for the dropped trees + the residuals
- This prevents any single tree from dominating and reduces overfitting

Key DART parameters:
- `boosting_type='dart'`
- `drop_rate`: fraction of trees to drop (default 0.1)
- `skip_drop`: probability of skipping dropout (default 0.5)

**Warning:** DART does NOT support early stopping — you must set `n_estimators` manually.

1. Train DART model with `drop_rate=0.1` and `skip_drop=0.5`.
2. Compare to standard `gbdt` on test AUC and training time.
3. Return `dart_comparison` DataFrame.

In [ ]:
# YOUR CODE HERE
dart_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert dart_comparison is not None
assert len(dart_comparison) == 2
assert set(dart_comparison['boosting_type']) == {'gbdt', 'dart'}
assert (dart_comparison['test_auc'] > 0.60).all()
print("✓ Exercise 4 passed")
print(dart_comparison.to_string(index=False))

---
## Exercise 5 — Multiclass Classification

**Task:** Use LightGBM for multiclass prediction on the iris dataset, then build your own confusion matrix from scratch.

1. Load iris dataset, train `LGBMClassifier` with `objective='multiclass'`, `num_class=3`.
2. Implement `confusion_matrix_from_scratch(y_true, y_pred)` using only numpy — no sklearn.
3. Implement `per_class_metrics(cm)` that computes precision, recall, F1 per class from the confusion matrix — again from scratch.
4. Compute macro-averaged F1 manually and verify it matches `sklearn.metrics.f1_score(..., average='macro')`.
5. Return `class_report`: DataFrame with class, precision, recall, f1, support.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.metrics import f1_score as sklearn_f1

iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_iris, y_iris, test_size=0.2, stratify=y_iris, random_state=42)

def confusion_matrix_from_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Build confusion matrix using only numpy.
    Returns (n_classes x n_classes) array.
    """
    # YOUR CODE HERE
    pass

def per_class_metrics(cm: np.ndarray) -> pd.DataFrame:
    """
    Compute precision, recall, F1 per class from confusion matrix.
    Returns DataFrame: class, precision, recall, f1, support
    """
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: train LGBMClassifier, predict, call functions
class_report = None

In [ ]:
# --- ASSERTIONS ---
assert class_report is not None
assert len(class_report) == 3
for col in ['precision', 'recall', 'f1', 'support']:
    assert col in class_report.columns

# Verify macro F1 against sklearn
lgbm_clf = lgb.LGBMClassifier(objective='multiclass', num_class=3, random_state=42, verbosity=-1)
lgbm_clf.fit(X_tr_i, y_tr_i)
y_pred_i = lgbm_clf.predict(X_te_i)
sklearn_macro = sklearn_f1(y_te_i, y_pred_i, average='macro')
scratch_macro = class_report['f1'].mean()
assert abs(sklearn_macro - scratch_macro) < 0.01, f"Macro F1 mismatch: {sklearn_macro:.4f} vs {scratch_macro:.4f}"

print(f"✓ Exercise 5 passed — Macro F1: {scratch_macro:.4f}")
print(class_report.to_string(index=False))

---
## Exercise 6 — Custom Loss Function

**Concept:** LightGBM lets you define a custom objective by providing the gradient and Hessian of your loss.

**Task:** Implement a **focal loss** objective — designed for imbalanced classification. It down-weights easy examples:

$$\text{FL}(p_t) = -(1 - p_t)^\gamma \log(p_t)$$

where $p_t = p$ if $y=1$ else $1-p$, and $\gamma$ is the focusing parameter (try $\gamma=2$).

The gradient and Hessian:
$$g_i = -(1-p_t)^\gamma (\gamma p_t \log(p_t) - (1-p_t)) \cdot \text{sign}$$
$$h_i = \text{(derive it — apply product rule to } g_i \text{)}$$

1. Implement `focal_loss_gradient(y_pred, dtrain, gamma=2.0)` returning `(grad, hess)`.
2. Train with `obj=focal_loss_gradient` using the native `lgb.train()` API.
3. Compare to standard binary cross-entropy on recall of the minority class.
4. Return `focal_comparison`: DataFrame comparing both.

In [ ]:
def focal_loss_gradient(y_pred: np.ndarray, dtrain: lgb.Dataset, gamma: float = 2.0):
    """
    Focal loss gradient and Hessian for LightGBM custom objective.
    y_pred: raw scores (before sigmoid)
    Returns: (grad, hess)
    """
    # YOUR CODE HERE
    # Step 1: convert raw scores to probabilities via sigmoid
    # Step 2: compute p_t per sample (p if y=1 else 1-p)
    # Step 3: compute gradient
    # Step 4: compute hessian
    pass

# YOUR CODE HERE: train both models, compare
focal_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert focal_comparison is not None
assert len(focal_comparison) == 2
assert 'recall' in focal_comparison.columns
print("✓ Exercise 6 passed")
print(focal_comparison.to_string(index=False))

---
## Exercise 7 — Feature Interaction Constraints

**Concept:** Feature interaction constraints limit which features can appear together in the same tree path — useful for interpretability and domain constraints.

Example: in credit risk, you might want to prevent `age` and `purpose` from interacting directly — only allow each to interact with financial features.

1. Define interaction constraints as a list of groups: `[[0,1,2], [3,4,5,6], [7,8,9]]` where each group is indices of features allowed to interact.
2. Train with `interaction_constraints` parameter.
3. Verify by inspecting `lgb_model.dump_model()` — check that no tree path contains features from different groups.
4. Compare test AUC: constrained vs unconstrained.
5. Return `constraint_report`: DataFrame with model, test_auc, n_trees.

In [ ]:
# Define groups based on feature index
n_features = len(feature_names)
group_size = n_features // 3
interaction_groups = [
    list(range(0, group_size)),
    list(range(group_size, 2*group_size)),
    list(range(2*group_size, n_features))
]

# YOUR CODE HERE
constraint_report = None

In [ ]:
# --- ASSERTIONS ---
assert constraint_report is not None
assert len(constraint_report) == 2
assert 'test_auc' in constraint_report.columns
print("✓ Exercise 7 passed")
print(constraint_report.to_string(index=False))

---
## Exercise 8 — Regression with LightGBM: Housing

**Task:** Full regression workflow comparing LightGBM objectives.

1. Train LightGBM on housing data with three objectives: `regression` (L2), `regression_l1` (L1), `huber`.
2. For each, compute RMSE, MAE, R², and **MAPE** (mean absolute percentage error).
3. **Huber loss** blends L1 and L2: uses L2 for small errors, L1 for large errors. The transition happens at `alpha` (default 0.9). Explain when Huber is preferable.
4. Return `regression_comparison`: DataFrame with objective, RMSE, MAE, R², MAPE.
5. Also: plot the residual distribution for each (use quantiles, not actual plot — output `residual_quantiles_df`).

In [ ]:
X_reg = housing.drop('medhousval', axis=1)
y_reg = housing['medhousval']
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# YOUR CODE HERE
regression_comparison = None
residual_quantiles_df = None

In [ ]:
# --- ASSERTIONS ---
assert regression_comparison is not None
assert len(regression_comparison) == 3
for col in ['rmse', 'mae', 'r2', 'mape']:
    assert col in regression_comparison.columns.str.lower().tolist()
r2_col = [c for c in regression_comparison.columns if 'r2' in c.lower()][0]
assert (regression_comparison[r2_col] > 0.7).all()
print("✓ Exercise 8 passed")
print(regression_comparison.to_string(index=False))

**Huber loss:** *(When is Huber loss preferable over pure L2 or L1? What kind of data distribution makes it the right choice?)*

---
## Exercise 9 — LightGBM vs XGBoost: Head-to-Head

**Task:** Systematic comparison on the credit dataset. Both models should be properly tuned.

1. Tune both models with `RandomizedSearchCV(n_iter=20)` over equivalent parameter spaces.
2. Compare on: test AUC, test F1, training time (seconds), prediction time (milliseconds), model size (bytes via `joblib`).
3. Build `comparison_df`: rows = [LightGBM, XGBoost], all metrics as columns.
4. Determine winner per metric and add a `winner` row.

**When to use each in practice:** answer in markdown after the assertions.

In [ ]:
import xgboost as xgb
import joblib, time, os

# YOUR CODE HERE
comparison_df = None

In [ ]:
# --- ASSERTIONS ---
assert comparison_df is not None
assert 'LightGBM' in comparison_df.index.tolist() or 'LightGBM' in comparison_df.values
assert 'XGBoost' in comparison_df.index.tolist() or 'XGBoost' in comparison_df.values
for col in ['test_auc', 'training_time_s']:
    assert any(col in c for c in comparison_df.columns.str.lower().tolist()), f"Missing: {col}"
print("✓ Exercise 9 passed")
print(comparison_df.to_string())

**When to use LightGBM vs XGBoost:**
- Use LightGBM when: *(your answer)*
- Use XGBoost when: *(your answer)*

---
## Exercise 10 — Capstone: End-to-End LightGBM Production Pipeline

**Spec:** Full production pipeline with LightGBM as the core model.

1. **Feature engineering**: add 5 domain-driven features (interactions, ratios, log transforms).
2. **LightGBM with native categoricals**: pass categoricals directly — no OHE.
3. **Nested CV**: outer 5-fold, inner 3-fold grid search — get unbiased performance estimate.
4. **Calibration**: use `CalibratedClassifierCV(method='isotonic')`.
5. **Threshold tuning**: optimize for F2 score (recall weighted 2× over precision) — useful when false negatives are more costly.
6. **SHAP global importance**: compute mean |SHAP| using `lgb.Dataset` and `lgb_model.predict(..., pred_contrib=True)`.
7. **Save**: serialize to `/tmp/lgbm_pipeline.joblib`.
8. Return `final_report` dict with all metrics + top 5 features by SHAP.

In [ ]:
from sklearn.metrics import fbeta_score

def f2_score(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

# YOUR CODE HERE
final_report = None

In [ ]:
# --- ASSERTIONS ---
import os
assert final_report is not None
required = ['nested_cv_auc_mean', 'nested_cv_auc_std', 'test_auc',
            'test_f2', 'optimal_threshold', 'top5_shap_features']
for k in required:
    assert k in final_report, f"Missing: {k}"
assert final_report['test_auc'] > 0.65
assert len(final_report['top5_shap_features']) == 5
assert os.path.exists('/tmp/lgbm_pipeline.joblib')

print("✓ Exercise 10 passed — Full LightGBM pipeline complete")
print(f"Nested CV AUC: {final_report['nested_cv_auc_mean']:.4f} ± {final_report['nested_cv_auc_std']:.4f}")
print(f"Test AUC: {final_report['test_auc']:.4f} | F2: {final_report['test_f2']:.4f}")
print(f"Top SHAP features: {final_report['top5_shap_features']}")